# 05 Train Full Core

Notebook này minh họa luồng `encoder -> retrieval -> history selection -> fusion` để bàn giao cho phần decoder/training.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(PROJECT_ROOT)

D:\KhaiPhaDuLieuLon\HealthDataMining


In [3]:
from pathlib import Path
import os
import sys
import json
import pyarrow.parquet as pq

# ===== Resolve project root safely =====
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("python =", sys.executable)
print("cwd =", Path.cwd())
print("PROJECT_ROOT =", PROJECT_ROOT)
print("src exists =", (PROJECT_ROOT / "src").exists())
print("handover_data exists =", (PROJECT_ROOT / "handover_data").exists())

from src.data.dataset import collate_batch
from src.models.patient_state_encoder import PatientStateEncoder
from src.models.history_selector import HistorySelector
from src.models.fusion import FusionModule
from src.retrieval.memory_bank import MemoryBank, build_last_visit_queries
from src.retrieval.topk_retriever import retrieve_topk



python = C:\Users\LEGION\.conda\envs\torch_py310\python.exe
cwd = D:\KhaiPhaDuLieuLon\HealthDataMining\notebooks
src exists = False


In [4]:
root = PROJECT_ROOT
handover_root = root / "handover_data"
processed_root = handover_root / "processed"
train_dir = processed_root / "train"
vocab_dir = handover_root / "vocab"

print("train_dir =", train_dir)
print("train_dir exists =", train_dir.exists())

if train_dir.exists():
    print("train files:")
    for p in sorted(train_dir.glob("*")):
        print(" -", p.name)

parquet_files = sorted(train_dir.glob("*.parquet"))
if not parquet_files:
    raise FileNotFoundError(f"Không tìm thấy file .parquet trong: {train_dir}")

sample_parquet = parquet_files[0]
print("Using parquet file:", sample_parquet.name)

records = pq.read_table(sample_parquet).slice(0, 4).to_pylist()

with (vocab_dir / "diagnosis_vocab.json").open("r", encoding="utf-8") as handle:
    diagnosis_vocab_size = json.load(handle)["size"]
with (vocab_dir / "procedure_vocab.json").open("r", encoding="utf-8") as handle:
    procedure_vocab_size = json.load(handle)["size"]
with (vocab_dir / "drug_vocab.json").open("r", encoding="utf-8") as handle:
    drug_vocab_size = json.load(handle)["size"]

print("diagnosis_vocab_size =", diagnosis_vocab_size)
print("procedure_vocab_size =", procedure_vocab_size)
print("drug_vocab_size =", drug_vocab_size)
print("num sample records =", len(records))
print("first record keys =", records[0].keys() if records else "no records")

batch = collate_batch(records)
encoder = PatientStateEncoder(
    diagnosis_vocab_size=diagnosis_vocab_size,
    procedure_vocab_size=procedure_vocab_size,
    drug_vocab_size=drug_vocab_size,
    num_lab_features=batch["lab_values"].shape[-1],
    num_vital_features=batch["vital_values"].shape[-1],
    code_embedding_dim=8,
    medication_embedding_dim=8,
    numeric_projection_dim=4,
    time_embedding_dim=4,
    visit_hidden_dim=16,
    hidden_dim=12,
    dropout=0.0,
)
selector = HistorySelector(hidden_dim=12, dropout=0.0)
fusion = FusionModule(hidden_dim=12, dropout=0.0)

encoder_outputs = encoder(batch)
bank = MemoryBank.build_from_batch(records, encoder_outputs, split="train")
query_states, query_metadata = build_last_visit_queries(records, encoder_outputs, split="train")
retrieval_payload = retrieve_topk(query_states, query_metadata, bank, top_k=2, temporal_decay_alpha=0.05)
selection_outputs = selector(
    current_state=encoder_outputs["pooled_state"],
    state_sequence=encoder_outputs["state_sequence"],
    visit_mask=encoder_outputs["visit_mask"],
    retrieval_payload=retrieval_payload,
    memory_bank=bank,
)
fusion_outputs = fusion(
    current_state=encoder_outputs["pooled_state"],
    self_history_context=selection_outputs["self_history_context"],
    neighbor_history_context=selection_outputs["neighbor_history_context"],
    branch_masks={
        "current": selection_outputs["evidence_metadata"]["self_history_available_mask"] | ~selection_outputs["evidence_metadata"]["self_history_available_mask"],
        "self": selection_outputs["evidence_metadata"]["self_history_available_mask"],
        "neighbor": selection_outputs["evidence_metadata"]["neighbor_available_mask"],
        "group": selection_outputs["evidence_metadata"]["group_available_mask"],
    },
)

encoder_outputs["pooled_state"].shape, fusion_outputs["fused_repr"].shape



FileNotFoundError: .../handover_data/processed/train/part-00000.parquet